# Agon Evaluation Analysis

This notebook compares the full debate system against two baselines on the 25-problem dataset in `data/problems.json`.

- `single_agent_baseline`: one direct answer per problem, with several fixed plausible mistakes.
- `majority_vote_baseline`: three independent generic solver answers are compared by majority vote.
- `full_debate`: saved debate outputs are used when `results/debate_runs.json` exists; otherwise `evaluation/baseline.py` uses a deterministic debate-synthesis answer. The Kant/Mill/Nietzsche/Camus role framing is used for the philosophical/ethical subset.


In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import SVG, Image, display

ROOT = Path('..').resolve()
results_dir = ROOT / 'results'
plots_dir = results_dir / 'plots'

summary = json.loads((results_dir / 'evaluation_summary.json').read_text(encoding='utf-8'))
rows = pd.read_csv(results_dir / 'evaluation_rows.csv')
rows.head()

## Dataset Overview

The current dataset contains 25 problems split across the five required categories.

In [ ]:
problem_data = json.loads((ROOT / 'data' / 'problems.json').read_text(encoding='utf-8'))
problems = problem_data['problems'] if isinstance(problem_data, dict) else problem_data
category_table = (
    pd.DataFrame(problems)
    .groupby('category')
    .agg(problem_count=('id', 'count'), first_problem=('id', 'min'), last_problem=('id', 'max'))
    .reset_index()
)
category_table

## Overall Results

The full debate system is compared directly to a single-agent baseline and a majority-vote baseline using accuracy and mean rubric score. Ethical role coverage is reported separately for the philosophical/ethical subset.

In [ ]:
system_table = pd.DataFrame(summary['systems']).T
system_table[['n', 'correct', 'accuracy', 'mean_score']]

In [ ]:
print(f"Improvement rate over single-agent baseline: {summary['improvement_rate']:.0%}")
print(f"Consensus rate across compared systems: {summary['consensus_rate']:.0%}")
print(f"Judge/full-debate accuracy: {summary['judge_accuracy']:.0%}")
print('Ethical role coverage:', summary.get('ethical_full_debate_role_coverage', {}))

## Category Breakdown

In [ ]:
category_rows = []
for category, systems in summary['categories'].items():
    for system, values in systems.items():
        category_rows.append({'category': category, 'system': system, **values})
pd.DataFrame(category_rows).sort_values(['category', 'system'])

## Generated Plots

The plots below are generated by `python -m evaluation.plots`. In environments with matplotlib they are saved as PNG files; otherwise the script writes SVG fallbacks.

In [ ]:
for stem in ['system_accuracy', 'category_scores', 'difficulty_accuracy', 'debate_rates']:
    png = plots_dir / f'{stem}.png'
    svg = plots_dir / f'{stem}.svg'
    print(stem)
    if png.exists():
        display(Image(filename=str(png)))
    elif svg.exists():
        display(SVG(filename=str(svg)))

## Interpretation

The latest evaluation uses 25 mixed problems: mathematical/logical reasoning, physics/scientific reasoning, logic puzzles or constraint satisfaction, strategic/game-theory reasoning, and gradeable philosophical/ethical reasoning. The deterministic full-debate row currently acts as a synthesis baseline: it checks direct calculations or rules for non-ethical problems, and uses the Kant/Mill/Nietzsche/Camus framing only for the ethical subset.

The current results show the full-debate synthesis above the single-agent baseline, while majority voting is also strong on many factual or calculation-heavy tasks. The clearest gains over the single-agent baseline come from problems where one direct answer makes a plausible shortcut error, such as probability, rectangle dimensions, bridge crossing, coin weighing, or backward induction.

For a final live experiment, run the real debate pipeline and save outputs to `results/debate_runs.json`; `evaluation/baseline.py` will automatically use those saved full-debate answers instead of the deterministic simulation.